# שבוע 7: ניתוח מטבעות מלא — מטלה 3

שבוע זה הוא שבוע הניתוח של **מטלה 3 — ניתוח צורת מטבעות רומיים**.

נשלב את כל מה שלמדנו:
1. טעינת נתוני TPS משני הקיסרים
2. GPA — יישור פרוקרוסטס
3. PCA — מרחב צורות
4. בדיקת מובהקות סטטיסטית (MANOVA בפרמוטציה)
5. השוואת צורות ממוצע בין הקיסרים

> **טיפ**: שמרו את תמונות הגרפים שלכם (`plt.savefig`) לדוח.

In [ ]:
!pip install morphops python-bidi -q
import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from bidi.algorithm import get_display
rtl = get_display
print('הכל מוכן!')

In [ ]:
import urllib.request
import morphops as mops

def parse_tps(text):
    specimens, ids = [], []
    lines = text.strip().split('\n')
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line.startswith('LM='):
            n_lm = int(line.split('=')[1])
            coords = []
            for j in range(n_lm):
                i += 1
                parts = lines[i].strip().replace(',', '.').split()
                coords.append([float(parts[0]), float(parts[1])])
            specimens.append(np.array(coords))
        elif line.startswith('ID='):
            ids.append(line.split('=')[1])
        i += 1
    return np.array(specimens), ids

def make_synthetic_coins(n, seed_offset=0):
    np.random.seed(42 + seed_offset)
    coins = []
    for i in range(n):
        cx, cy = np.random.uniform(100, 500), np.random.uniform(100, 500)
        scale = np.random.uniform(0.8, 1.2)
        ao = np.random.uniform(0, 0.2)
        lm = [[cx + (80*scale + np.random.randn()*3)*np.cos(2*np.pi*j/8 + ao),
               cy + (80*scale + np.random.randn()*3)*np.sin(2*np.pi*j/8 + ao)] for j in range(8)]
        coins.append(np.array(lm))
    return np.array(coins), [f'coin_{i+1:03d}' for i in range(n)]

base = 'https://raw.githubusercontent.com/shaigordin/comparch/2026/morphometrics/data/coins/'
try:
    with urllib.request.urlopen(base + 'hadrian.tps') as r:
        lm_h, ids_h = parse_tps(r.read().decode('utf-8'))
    with urllib.request.urlopen(base + 'antoninus.tps') as r:
        lm_a, ids_a = parse_tps(r.read().decode('utf-8'))
    print(f'הדריאנוס: {len(lm_h)} | אנטונינוס פיוס: {len(lm_a)}')
except Exception as e:
    print(f'משתמשים בנתוני דוגמה: {e}')
    lm_h, ids_h = make_synthetic_coins(20, 0)
    lm_a, ids_a = make_synthetic_coins(15, 10)

all_lm = np.concatenate([lm_h, lm_a])
labels = np.array(['הדריאנוס'] * len(lm_h) + ['אנטונינוס פיוס'] * len(lm_a))
aligned, mean_shape, _ = mops.procrustes(all_lm)
print(f'GPA הושלם: {len(aligned)} מטבעות')

In [ ]:
from sklearn.decomposition import PCA

X = aligned.reshape(len(aligned), -1)
pca = PCA()
scores = pca.fit_transform(X)
var = pca.explained_variance_ratio_ * 100

colors = {'הדריאנוס': '#2196F3', 'אנטונינוס פיוס': '#FF5722'}
fig, ax = plt.subplots(figsize=(9, 7))

for group, color in colors.items():
    mask = labels == group
    ax.scatter(scores[mask, 0], scores[mask, 1], c=color, s=90, alpha=0.8,
               label=rtl(group), edgecolors='white', linewidth=0.5)

ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel(rtl(f'PC1 ({var[0]:.1f}% שונות)'))
ax.set_ylabel(rtl(f'PC2 ({var[1]:.1f}% שונות)'))
ax.set_title(rtl('מרחב צורות — מטבעות הדריאנוס ואנטונינוס פיוס'), fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('coins_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print('הגרף נשמר: coins_pca.png')

## בדיקת מובהקות סטטיסטית

האם ההפרדה בין הקבוצות מובהקת סטטיסטית?
נשתמש ב-**MANOVA בשיטת פרמוטציה** (999 ערבולים אקראיים):

- **H₀**: אין הבדל בצורה בין הדריאנוס לאנטונינוס פיוס
- **H₁**: יש הבדל מובהק בצורה

In [ ]:
def permutation_manova(X, groups, n_perm=999, seed=42):
    np.random.seed(seed)
    def f_stat(X, g):
        unique_g = np.unique(g)
        gm = X.mean(axis=0)
        between = sum(np.sum(g==u) * np.sum((X[g==u].mean(0) - gm)**2) for u in unique_g)
        within  = sum(np.sum((X[g==u] - X[g==u].mean(0))**2) for u in unique_g)
        return between / within if within > 0 else 0

    obs = f_stat(X, groups)
    perm = [f_stat(X, np.random.permutation(groups)) for _ in range(n_perm)]
    p = (np.sum(np.array(perm) >= obs) + 1) / (n_perm + 1)
    r2 = obs / (obs + 1)
    return obs, p, r2

f_stat, p_val, r2 = permutation_manova(scores[:, :4], labels)
print(f'תוצאות MANOVA (פרמוטציה, 999 ערבולים):')
print(f'  F = {f_stat:.3f}')
print(f'  p = {p_val:.3f}')
print(f'  R² = {r2:.3f} ({r2*100:.1f}% שונות מוסברת)')
print()
if p_val < 0.05:
    print('→ ההפרדה מובהקת סטטיסטית (p < 0.05)')
else:
    print('→ ההפרדה אינה מובהקת סטטיסטית (p ≥ 0.05)')

## ויזואליזציית צורה: צורות ממוצע לכל קיסר

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
group_colors = {'הדריאנוס': '#2196F3', 'אנטונינוס פיוס': '#FF5722'}

for ax, (group, color) in zip(axes, group_colors.items()):
    mask = labels == group
    mean_g = aligned[mask].mean(axis=0)
    ax.plot(np.append(mean_g[:, 0], mean_g[0, 0]),
            np.append(mean_g[:, 1], mean_g[0, 1]),
            'o-', color=color, markersize=10, linewidth=2)
    for j, (x, y) in enumerate(mean_g):
        ax.annotate(str(j+1), (x, y), textcoords='offset points', xytext=(5, 5), fontsize=10)
    ax.set_title(rtl(f'צורת ממוצע — {group} (n={mask.sum()})'), fontsize=12)
    ax.set_aspect('equal')
    ax.set_xlim(-0.45, 0.45)
    ax.set_ylim(-0.45, 0.45)
    ax.grid(True, alpha=0.3)

plt.suptitle(rtl('השוואת צורות ממוצע בין הקיסרים'), fontsize=14)
plt.tight_layout()
plt.savefig('coins_mean_shapes.png', dpi=150, bbox_inches='tight')
plt.show()
print('הגרף נשמר: coins_mean_shapes.png')

## סיכום תוצאות לדוח

כתבו בדוח:

| פרמטר | ערך |
|--------|-----|
| מספר מטבעות | |
| PC1 שונות % | |
| PC2 שונות % | |
| F (MANOVA) | |
| p-value | |
| R² | |

## שאלות לדיון

1. **שאלת מחקר**: האם קיים הבדל סטטיסטי מובהק בצורת המטבע?
2. **פרשנות ארכאולוגית**: מה ההבדל הגיאומטרי בין הקיסרים? מה אפשר ללמוד על מנהגי הטביעה?
3. **מגבלות**: מה עלול להסביר שונות בתוך הקבוצות (שנות שלטון שונות, מנטות שונות)?